This is a project to practice webscraping using Scrapy. Our goal today is to implement a scraper to get data about the actors in the user's favorite movie or TV show, and to use that data to give recommendations of other movies or TV shows with those same actors. 

A scraper uses a special class called a `spider`. We will be writing the code for our class `tmdb_spider`.


## Let's Get Started!

Open a terminal and navigate to the location where you would like to put your project (mine is in my `Projects` folder). Don't forget to activate the environment you want! Then, type:

```bash
scrapy startproject TMDB_scraper
```

This should create a folder called `TMDB_scraper` in your current directory. Inside this folder, you should have another folder called `TMDB_scraper` and a file called `scrapy.cfg`. Inside of `TMDB_scraper`, you should have some `.py` files, along with another folder called `spiders`. Make a file called `tmdb_spider.py` -- the code to implement our `tmdb_spider` class will live in here.

# The website:
Before building a scraper, get to know the website we'll be scraping a little bit. The example I'm using is *Harry Potter and the Philosopher's Stone*.
https://www.themoviedb.org/
Search for your favorite movie and find its page, or follow along with *Harry Potter*.

Take a look at the page, and try opening your Web Developer tools to get an idea of the `html` that makes up this website.

Click on the "Full Cast & Crew" button to go to the cast page. Then click on one of the actors and find a list of all the works they've acted in. This is what we want our spiders to do for us.


# Initial Code:
The first things we need to put in this file are the following:

```python
import scrapy

class TmdbSpider(scrapy.Spider):
    name = 'tmdb_spider'
    def __init__(self, subdir="", *args, **kwargs):
        self.start_urls = [f"https://www.themoviedb.org/movie/{subdir}/"]
```
Here we just imported `scrapy` and initialized our class -- note that it is a child class of `scrapy.Spider`! The arguments that the initializer takes make it so that we can use this class on different starting sites, which makes it so that this works for different movies (the user's favorite movie is taken as input when they run it from the terminal).



# Methods
We'll be implementing three methods for our spider: 

- `parse()`
- `parse_full_credits()`
- `parse_actor_page()`
    
Let's go through how each one works.

### The `parse()` Method:
This method takes as an argument a scrapy `Response` object and `yield`s a scrapy `Request` object. We will assume that the given `Response` argument is the starting webpage for the user's favorite movie. `parse()` will send a `Request` to the cast page and then call our next method. Here is the code:
```python
    def parse(self, response):
        cast_page = response.url + "/cast"
        yield response.follow(cast_page, callback=self.parse_full_credits)
```

### The `parse_full_credits()` Method:
This method also takes a `Response` object as an argument -- this time it's the cast page. This method makes a list of the actors in our movie (I put it in a `set` to avoid duplicates) and then calls the next method for each actor's page. Here is the code:
```python
    def parse_full_credits(self, response):
        actor_pages = response.css('ol:not(.crew) a::attr(href)').getall()
        for actor_page in set(actor_pages):
            yield response.follow(actor_page, callback=self.parse_actor_page)
```

Here's how it works:

- `response.css` is a built-in `scrapy` function that takes in a `string` which is a css command
- This command means to select all of the `ol` items that are not of the `crew` class (because we only want to select cast members, not crew members)
- Then within the `ol` items select all the `a` items' `href` attribute
- The `href` attribute is the link to the actor's page. You can find it by searching in the website's `html` using your Developer Tools
- We then put it in a `set` to avoid duplicates
- Finally, iterate through this set and send a `Request` to each actor's page, calling `parse_actor_page()`

### The `parse_actor_page()` Method:
This method also takes a `Response` object as an argument -- the actor's page. This method then yields a dictionary containing the actor's name and each movie/TV show that they have acted in. Here is the code:
```python
    def parse_actor_page(self, response):
        actor_name = response.css('head title::text').get()
        actor_name = actor_name.replace(' — The Movie Database (TMDB)', '')
        titles = response.xpath(
            "//h3[contains(text(), 'Acting')]/following-sibling::table[1]//bdi//text()"
            ).extract()
        for title in set(titles):
            yield {"actor" : actor_name, "movie_or_TV_name" : title}
```

Here's how it works:

- We use `response.css()` again to get the actor's name -- here it is stored as a `title` object within the `head` object
- Since this method returns a string containing the actor's name followed by ' - The Movie Database (TMDB)', we use the `replace` method to drop that
- Since css doesn't have a `contains` method, we have to use Xpath for this part. Luckily, `scrapy` also has a built-in method for Xpath. Here we are using it to extract only the names of movies/shows that the actor has *acted* in
- Then it yields a `dict` with the actor's name and the movie/show name


# Next Steps

When you are ready to implement the spider, open up the terminal again and run:

```bash
scrapy crawl tmdb_spider -o results.csv -a subdir=671-harry-potter-and-the-philosopher-s-stone
```

You can replace the subdir argument with a different movie if you prefer. This command will create a `csv` file in this directory called `results.csv` which contains the actors' names and the names of the movies/shows they've been in. 

### Displaying the Data

I prefer to display this data in a DataFrame or plotly chart using Jupyter Notebook.

In [1]:
import pandas as pd
import numpy as np
import plotly
import plotly.io as pio
pio.renderers.default="iframe"
from plotly import express as px

In [2]:
movies = pd.read_csv('results.csv')
movies.head()

,actor,movie_or_TV_name
0,Warwick Davis,Harry Potter and the Order of the Phoenix
1,Warwick Davis,The Fast Show
2,Warwick Davis,When Star Wars Ruled the World
3,Warwick Davis,M.I. High
4,Warwick Davis,Leprechaun 4: In Space


We'll use `transform()` to make another column containing the number of shared actors that each movie/show has:

In [3]:
movies['Shared Actors'] = movies.groupby('movie_or_TV_name').transform('count')

In [4]:
movies.head()

,actor,movie_or_TV_name,Shared Actors
0,Warwick Davis,Harry Potter and the Order of the Phoenix,24
1,Warwick Davis,The Fast Show,1
2,Warwick Davis,When Star Wars Ruled the World,1
3,Warwick Davis,M.I. High,1
4,Warwick Davis,Leprechaun 4: In Space,1


In [5]:
movies.sort_values(by = 'Shared Actors', inplace = True, ascending = False, ignore_index = True)
movies

,actor,movie_or_TV_name,Shared Actors
0,Matthew Lewis,Harry Potter and the Philosopher's Stone,62
1,Adrian Rawlins,Harry Potter and the Philosopher's Stone,62
2,Alan Rickman,Harry Potter and the Philosopher's Stone,62
3,Leila Hoffman,Harry Potter and the Philosopher's Stone,62
4,Eleanor Columbus,Harry Potter and the Philosopher's Stone,62
...,...,...,...
3076,Nina Young,The Magnificent Eleven,1
3077,Nina Young,The Surfer,1
3078,Nina Young,Molly Moon and the Incredible Book of Hypnotism,1
3079,Nina Young,Out of the Ashes,1


In [6]:
recs = movies[['movie_or_TV_name', 'Shared Actors']].drop_duplicates(
    subset = 'movie_or_TV_name', ignore_index = True)
recs = recs.head(15)
recs.rename(columns = {"Shared Actors" : "# of Shared Actors", 
                       "movie_or_TV_name" : "Movie/TV Name"}, 
                        inplace = True)
recs

,Movie/TV Name,# of Shared Actors
0,Harry Potter and the Philosopher's Stone,62
1,Harry Potter and the Chamber of Secrets,37
2,Harry Potter and the Prisoner of Azkaban,26
3,Harry Potter and the Order of the Phoenix,24
4,Harry Potter and the Deathly Hallows: Part 2,23
5,Harry Potter and the Deathly Hallows: Part 1,20
6,Harry Potter and the Goblet of Fire,19
7,Harry Potter and the Half-Blood Prince,19
8,Doctor Who,13
9,Harry Potter 20th Anniversary: Return to Hogwarts,11


In [7]:
fig = px.histogram(recs,
                   x = "Movie/TV Name",
                   y = "# of Shared Actors",
                   color = "Movie/TV Name",
                   hover_name = "Movie/TV Name",
                   opacity = 0.7,
                   nbins = 30)

# reduce whitespace
fig.update_layout(margin={"r":0,"t":50,"l":0,"b":0})
# show the plot
fig.show()